# RNN-based Legal Text Summarization

A minimal implementation of sequence-to-sequence summarization using GRU-based encoder-decoder architecture.

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import json
import random

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

## 2. Data Loading

In [ ]:
# Load the legal summarization dataset
with open('all_v1.json', 'r') as f:
    data = json.load(f)

# Extract original_text and reference_summary pairs
texts = []
summaries = []

for key, item in data.items():
    if 'original_text' in item and 'reference_summary' in item:
        texts.append(item['original_text'])
        summaries.append(item['reference_summary'])

print(f'Loaded {len(texts)} text-summary pairs')
print(f'\nSample text: {texts[0][:200]}...')
print(f'\nSample summary: {summaries[0]}')

## 3. Text Preprocessing

In [ ]:
# Special tokens
PAD_TOKEN = '<PAD>'
SOS_TOKEN = '<SOS>'
EOS_TOKEN = '<EOS>'
UNK_TOKEN = '<UNK>'

class Vocabulary:
    """Vocabulary class for mapping words to indices and vice versa."""
    
    def __init__(self):
        self.word2idx = {PAD_TOKEN: 0, SOS_TOKEN: 1, EOS_TOKEN: 2, UNK_TOKEN: 3}
        self.idx2word = {0: PAD_TOKEN, 1: SOS_TOKEN, 2: EOS_TOKEN, 3: UNK_TOKEN}
        self.word_count = {}
        self.n_words = 4  # Start after special tokens
    
    def add_sentence(self, sentence):
        """Add all words in a sentence to the vocabulary."""
        for word in sentence.split():
            self.add_word(word)
    
    def add_word(self, word):
        """Add a word to the vocabulary."""
        if word not in self.word2idx:
            self.word2idx[word] = self.n_words
            self.idx2word[self.n_words] = word
            self.word_count[word] = 1
            self.n_words += 1
        else:
            self.word_count[word] = self.word_count.get(word, 0) + 1
    
    def __len__(self):
        return self.n_words


def tokenize(text):
    """Simple tokenization: lowercase and split on whitespace."""
    return text.lower().split()


def text_to_indices(text, vocab, max_len=None):
    """Convert text to list of indices."""
    tokens = tokenize(text)
    indices = [vocab.word2idx.get(token, vocab.word2idx[UNK_TOKEN]) for token in tokens]
    if max_len:
        indices = indices[:max_len]
    return indices


def pad_sequence(seq, max_len, pad_idx=0):
    """Pad sequence to max_len."""
    if len(seq) < max_len:
        seq = seq + [pad_idx] * (max_len - len(seq))
    return seq[:max_len]


# Build vocabulary from all texts and summaries
vocab = Vocabulary()

for text in texts:
    vocab.add_sentence(text.lower())
for summary in summaries:
    vocab.add_sentence(summary.lower())

print(f'Vocabulary size: {len(vocab)}')

## 4. Dataset and DataLoader

In [ ]:
class LegalSummarizationDataset(Dataset):
    """PyTorch Dataset for legal text summarization."""
    
    def __init__(self, texts, summaries, vocab, max_src_len=200, max_tgt_len=50):
        self.texts = texts
        self.summaries = summaries
        self.vocab = vocab
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        # Convert text to indices
        src_indices = text_to_indices(self.texts[idx], self.vocab, self.max_src_len)
        src_indices = pad_sequence(src_indices, self.max_src_len, self.vocab.word2idx[PAD_TOKEN])
        
        # Convert summary to indices with SOS and EOS tokens
        tgt_indices = [self.vocab.word2idx[SOS_TOKEN]]
        tgt_indices += text_to_indices(self.summaries[idx], self.vocab, self.max_tgt_len - 2)
        tgt_indices.append(self.vocab.word2idx[EOS_TOKEN])
        tgt_indices = pad_sequence(tgt_indices, self.max_tgt_len, self.vocab.word2idx[PAD_TOKEN])
        
        return (
            torch.tensor(src_indices, dtype=torch.long),
            torch.tensor(tgt_indices, dtype=torch.long)
        )


# Create dataset and dataloader
MAX_SRC_LEN = 200
MAX_TGT_LEN = 50
BATCH_SIZE = 16

dataset = LegalSummarizationDataset(texts, summaries, vocab, MAX_SRC_LEN, MAX_TGT_LEN)

# Split into train and test
train_size = int(0.9 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f'Train samples: {len(train_dataset)}')
print(f'Test samples: {len(test_dataset)}')

## 5. RNN Model (Encoder-Decoder with GRU)

In [ ]:
class Encoder(nn.Module):
    """GRU-based encoder for sequence-to-sequence model."""
    
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=1, dropout=0.1):
        super(Encoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.gru = nn.GRU(embed_size, hidden_size, num_layers, 
                         batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        # x: (batch_size, seq_len)
        embedded = self.dropout(self.embedding(x))  # (batch_size, seq_len, embed_size)
        outputs, hidden = self.gru(embedded)  # outputs: (batch_size, seq_len, hidden_size)
        return outputs, hidden


class Decoder(nn.Module):
    """GRU-based decoder for sequence-to-sequence model."""
    
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=1, dropout=0.1):
        super(Decoder, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.gru = nn.GRU(embed_size, hidden_size, num_layers,
                         batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, hidden):
        # x: (batch_size, 1)
        embedded = self.dropout(self.embedding(x))  # (batch_size, 1, embed_size)
        output, hidden = self.gru(embedded, hidden)  # output: (batch_size, 1, hidden_size)
        prediction = self.fc(output.squeeze(1))  # (batch_size, vocab_size)
        return prediction, hidden


class Seq2Seq(nn.Module):
    """Sequence-to-sequence model combining encoder and decoder."""
    
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    
    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        tgt_len = tgt.shape[1]
        vocab_size = self.decoder.fc.out_features
        
        # Store outputs
        outputs = torch.zeros(batch_size, tgt_len, vocab_size).to(self.device)
        
        # Encode source
        _, hidden = self.encoder(src)
        
        # First input is SOS token
        decoder_input = tgt[:, 0].unsqueeze(1)  # (batch_size, 1)
        
        for t in range(1, tgt_len):
            output, hidden = self.decoder(decoder_input, hidden)
            outputs[:, t, :] = output
            
            # Teacher forcing
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1).unsqueeze(1)
            decoder_input = tgt[:, t].unsqueeze(1) if teacher_force else top1
        
        return outputs


# Model hyperparameters
EMBED_SIZE = 128
HIDDEN_SIZE = 256
NUM_LAYERS = 1
DROPOUT = 0.1

# Initialize model
encoder = Encoder(len(vocab), EMBED_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT)
decoder = Decoder(len(vocab), EMBED_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT)
model = Seq2Seq(encoder, decoder, device).to(device)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

## 6. Training Loop

In [ ]:
def train_epoch(model, dataloader, optimizer, criterion, device, teacher_forcing_ratio=0.5):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    
    for src, tgt in dataloader:
        src, tgt = src.to(device), tgt.to(device)
        
        optimizer.zero_grad()
        output = model(src, tgt, teacher_forcing_ratio)
        
        # Reshape for loss calculation (ignore first token which is SOS)
        output = output[:, 1:, :].reshape(-1, output.shape[-1])
        tgt = tgt[:, 1:].reshape(-1)
        
        loss = criterion(output, tgt)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        total_loss += loss.item()
    
    return total_loss / len(dataloader)


def evaluate(model, dataloader, criterion, device):
    """Evaluate the model."""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for src, tgt in dataloader:
            src, tgt = src.to(device), tgt.to(device)
            
            output = model(src, tgt, teacher_forcing_ratio=0)
            
            output = output[:, 1:, :].reshape(-1, output.shape[-1])
            tgt = tgt[:, 1:].reshape(-1)
            
            loss = criterion(output, tgt)
            total_loss += loss.item()
    
    return total_loss / len(dataloader)


# Training setup
LEARNING_RATE = 0.001
NUM_EPOCHS = 10

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=vocab.word2idx[PAD_TOKEN])

# Training loop
train_losses = []
test_losses = []

print('Starting training...')
for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    test_loss = evaluate(model, test_loader, criterion, device)
    
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    
    print(f'Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Test Loss: {test_loss:.4f}')

print('Training complete!')

## 7. Inference

In [ ]:
def generate_summary(model, text, vocab, max_len=50, device='cpu'):
    """Generate a summary for the given text."""
    model.eval()
    
    # Preprocess input text
    src_indices = text_to_indices(text, vocab, MAX_SRC_LEN)
    src_indices = pad_sequence(src_indices, MAX_SRC_LEN, vocab.word2idx[PAD_TOKEN])
    src_tensor = torch.tensor([src_indices], dtype=torch.long).to(device)
    
    with torch.no_grad():
        # Encode
        _, hidden = model.encoder(src_tensor)
        
        # Start with SOS token
        decoder_input = torch.tensor([[vocab.word2idx[SOS_TOKEN]]], dtype=torch.long).to(device)
        
        generated_tokens = []
        
        for _ in range(max_len):
            output, hidden = model.decoder(decoder_input, hidden)
            top1 = output.argmax(1).item()
            
            if top1 == vocab.word2idx[EOS_TOKEN]:
                break
            
            if top1 != vocab.word2idx[PAD_TOKEN]:
                generated_tokens.append(vocab.idx2word[top1])
            
            decoder_input = torch.tensor([[top1]], dtype=torch.long).to(device)
    
    return ' '.join(generated_tokens)


# Test inference on a sample
sample_text = texts[0]
generated = generate_summary(model, sample_text, vocab, device=device)
print(f'Input text: {sample_text[:200]}...')
print(f'\nGenerated summary: {generated}')
print(f'\nReference summary: {summaries[0]}')

## 8. Demo - Sample Predictions

In [ ]:
# Generate summaries for a few test samples
print('=' * 80)
print('DEMO: Sample Predictions on Test Data')
print('=' * 80)

# Get some test indices
test_indices = list(test_dataset.indices)[:5]

for i, idx in enumerate(test_indices):
    text = texts[idx]
    reference = summaries[idx]
    generated = generate_summary(model, text, vocab, device=device)
    
    print(f'\n--- Sample {i+1} ---')
    print(f'Input (truncated): {text[:150]}...')
    print(f'Reference: {reference}')
    print(f'Generated: {generated}')

print('\n' + '=' * 80)
print('Demo complete!')